In [12]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver  # , for saving state into ram memory
from langchain_groq import ChatGroq



In [13]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [14]:
model = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [15]:
class JokeState(TypedDict):

    topic:str
    joke:str
    explanation:str
    

In [16]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [19]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase "feeling a little crusty" to describe its bad mood. The joke relies on this wordplay to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The unexpected twist on the usual meaning of "crusty" creates the comedic effect.'}

In [20]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase "feeling a little crusty" to describe its bad mood. The joke relies on this wordplay to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The unexpected twist on the usual meaning of "crusty" creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase "feeling a little crusty" to describe its bad mood. The joke relies on this wordplay to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The unexpected twist on the usual meaning of "crusty" creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

In [24]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.',
 'explanation': 'This joke is a play on words that uses a common phrase associated with relationships and gives it a literal twist. The phrase "tangled up" typically refers to becoming deeply involved or entangled in a situation, often in a way that\'s complicated or difficult to escape. In the context of marriage, it might mean getting caught up in the responsibilities, emotions, and complexities that come with a lifelong commitment.\n\nHowever, in this joke, "tangled up" takes on a literal meaning because spaghetti is a type of long, thin, twisted pasta that can easily become tangled. The joke relies on this double meaning to create humor. The punchline suggests that the spaghetti is afraid of getting married because it\'s afraid of becoming literally tangled up, but the phrase also references the common fear of getting entangled in a co